In [ ]:
!pip install deepface opencv-python pandas

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile

with zipfile.ZipFile("dataset.zip", 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
import os

print(os.listdir("dataset"))

In [ ]:
from deepface import DeepFace
import numpy as np
import os

In [ ]:
database = {}
dataset_path = "dataset"

In [ ]:
# Loop through each person (folder)
for person in os.listdir(dataset_path):
    person_path = os.path.join(dataset_path, person)

    # Skip if not a folder
    if not os.path.isdir(person_path):
        continue

    embeddings = []

    # Loop through each image of that person
    for img_name in os.listdir(person_path):
        img_path = os.path.join(person_path, img_name)

        try:
            embedding = DeepFace.represent(
                img_path=img_path,
                enforce_detection=False   # IMPORTANT FIX
            )[0]["embedding"]

            embeddings.append(embedding)

        except Exception as e:
            print(f"❌ Error with {img_path}: {e}")

    # Create average embedding for that person
    if len(embeddings) > 0:
        avg_embedding = np.mean(embeddings, axis=0)
        database[person] = avg_embedding
        print(f"✅ Added {person} with {len(embeddings)} images")

    else:
        print(f"⚠️ No valid images for {person}")

# Final check
print("\n🎯 Database built with people:")
print(database.keys())

In [ ]:
import numpy as np

def recognize_face(img_path):
    query_embedding = DeepFace.represent(
        img_path=img_path,
        enforce_detection=False
    )[0]["embedding"]

    min_distance = float("inf")
    identity = "Unknown"

    for name, db_embedding in database.items():
        distance = np.linalg.norm(
            np.array(query_embedding) - np.array(db_embedding)
        )

        if distance < min_distance:
            min_distance = distance
            identity = name

    # threshold (we can tune later)
    if min_distance < 10:
        return identity, min_distance
    else:
        return "Unknown", min_distance

In [ ]:
import os
print(os.listdir("dataset/artist2"))

In [ ]:
name, dist = recognize_face("dataset/artist2/artist2_pic4.png")

In [ ]:
print("Recognized:", name)
print("Distance:", dist)

In [ ]:
import pandas as pd
from datetime import datetime

attendance = pd.DataFrame(columns=["Name", "Time"])

In [ ]:
def mark_attendance(name):
    global attendance

    # avoid duplicate
    if name not in attendance["Name"].values:
        time_now = datetime.now().strftime("%H:%M:%S")
        attendance.loc[len(attendance)] = [name, time_now]

In [ ]:
name, dist = recognize_face("dataset/artist2/artist2_pic4.png")

print("Recognized:", name)

if name != "Unknown":
    mark_attendance(name)

attendance

In [ ]:
attendance.to_csv("attendance.csv", index=False)

In [ ]:
from google.colab import files
files.download("attendance.csv")